# NCAA Reward Farmer
Run cells top to bottom. Re-run any cell anytime to change settings.

In [ ]:
# ── Cell 1: Setup ───────────────────────────────────────────
import os, sys
sys.path.insert(0, os.path.abspath(''))

from predx.config import load_dotenv_if_present
load_dotenv_if_present()

from predx.tools.reward_farmer import FarmerConfig

# ── MODE ──
DRY_RUN = False              # True = print only, no orders
TESTING = True               # True = small size (10 shares), ignore rewards floor
TESTING_SIZE = 10.0           # size when TESTING=True

# ── PRODUCTION SETTINGS (used when TESTING=False) ──
ORDER_SIZE = 1000.0           # contracts per side (min 1000 for rewards)
MIN_REWARD = 5000             # minimum $/day to farm
DB_PATH = 'trades.db'

# ── PRICE RANGE ──
MIN_PRICE = 0.01              # don't quote below this
MAX_PRICE = 0.99              # don't quote above this

# ── POSITION ──
MAX_POSITION = 50.0           # max contracts per side per market
SKEW_THRESHOLD = 99999.0      # disabled
SKEW_FACTOR = 0.5             # disabled
PREGAME_MULT = 1.0
LIVE_MULT = 1.0

# ── OBI / TP / SL ──
OBI_MIN_RATIO = 0.5           # don't BUY if bid_depth/ask_depth < this
TAKE_PROFIT_PCT = 2.0         # SELL at avg_price + this %
STOP_LOSS_PCT = 1.0           # dump if mid drops this % below avg

cfg = FarmerConfig(
    order_size=ORDER_SIZE,
    dry_run=DRY_RUN,
    testing=TESTING,
    testing_size=TESTING_SIZE,
    db_path=DB_PATH,
    min_price=MIN_PRICE,
    max_price=MAX_PRICE,
    max_position=MAX_POSITION,
    skew_threshold=SKEW_THRESHOLD,
    skew_factor=SKEW_FACTOR,
    pregame_size_mult=PREGAME_MULT,
    live_size_mult=LIVE_MULT,
    obi_min_ratio=OBI_MIN_RATIO,
    take_profit_pct=TAKE_PROFIT_PCT,
    stop_loss_pct=STOP_LOSS_PCT,
)

mode = "TESTING (10 shares)" if TESTING else f"LIVE ({ORDER_SIZE} shares)"
if DRY_RUN: mode = "DRY RUN"
print(f'Mode: {mode}')
print(f'  price_range={MIN_PRICE}-{MAX_PRICE}  max_pos={MAX_POSITION}')
print(f'  OBI_gate={OBI_MIN_RATIO}  TP={TAKE_PROFIT_PCT}%  SL={STOP_LOSS_PCT}%')

In [ ]:
# ── Cell 2: Discover markets ────────────────────────────────
from farmer import discover_markets, print_markets

markets = discover_markets(min_reward=MIN_REWARD)
print_markets(markets)

In [ ]:
# ── Cell 3: Pick markets ────────────────────────────────────
# Default: all on. Uncomment below to pick specific ones.

picks = {i: True for i in range(len(markets))}

# ── UNCOMMENT TO PICK SPECIFIC MARKETS ──
# picks = {i: False for i in range(len(markets))}
# picks[0] = True   # turn on just the ones you want

for i, m in enumerate(markets):
    status = '✓' if picks.get(i) else '✗'
    print(f'  [{i}] {status}  ${m["_rate"]:>7}/day  {m["question"][:55]}')

selected = [markets[i] for i in picks if picks[i]]
selected_cids = [m['condition_id'] for m in selected]
print(f'\n{len(selected_cids)} market(s) selected')

In [ ]:
# ── Cell 4: Start bot (background thread) ──────────────────
from predx import PolymarketClient
from predx.tools.reward_farmer import resolve_pair, BotRunner

pm = PolymarketClient()
pairs = []
for i, cid in enumerate(selected_cids):
    try:
        # Pass market_data if available from discovery
        md = selected[i] if i < len(selected) else None
        pair = resolve_pair(pm, cid, None, market_data=md)
        pairs.append(pair)
        game = pair.game_start_time or "unknown"
        print(f'  ✓ {pair.label[:45]}  tick={pair.tick_size}  reward=${pair.rewards_daily_rate}/day  game={game}')
    except Exception as e:
        print(f'  ✗ {cid}: {e}')
pm.close()

runner = BotRunner(pairs, cfg)
runner.start()

In [ ]:
# ── Cell 5: Check bot status (run anytime) ─────────────────
runner.status()

In [ ]:
# ── Cell 6: Change settings mid-run ────────────────────────
# All changes take effect on next requote cycle

cfg.order_size = 1000.0          # change size
cfg.max_position = 5000.0        # change position cap
cfg.merge_threshold = 5.0        # merge when min(yes, no) >= this
cfg.obi_min_ratio = 0.5          # OBI gate threshold
cfg.take_profit_pct = 2.0        # TP at avg + this %
cfg.stop_loss_pct = 3.0          # SL if mid drops this % below avg
# cfg.dry_run = False            # go live (uncomment when ready)

print(f'Updated: size={cfg.order_size}  dry_run={cfg.dry_run}')
print(f'  OBI={cfg.obi_min_ratio}  TP={cfg.take_profit_pct}%  SL={cfg.stop_loss_pct}%  merge@={cfg.merge_threshold}')

In [ ]:
# ── Cell 7: Stop bot ────────────────────────────────────────
runner.stop()

In [ ]:
# ── Cell 8: Check trade tape ────────────────────────────────
import sqlite3

db = sqlite3.connect(DB_PATH)
rows = db.execute('SELECT dt, market, side, price, size FROM trades ORDER BY ts DESC LIMIT 20').fetchall()
db.close()

if rows:
    print(f'{"Time":>24}  {"Side":<10}  {"Price":>6}  {"Size":>8}  Market')
    print('─' * 75)
    for dt, market, side, price, size in rows:
        print(f'{dt:>24}  {side:<10}  {price:>6.3f}  {size:>8.0f}  {market[:20]}...')
else:
    print('No trades yet.')

In [ ]:
# ── Cell 9: Check open orders ───────────────────────────────
from predx import PolymarketClient
pm = PolymarketClient()
try:
    orders = pm.get_open_orders()
    print(f'{len(orders)} open orders')
    for o in orders[:10]:
        print(f'  {o.get("side")}  {o.get("original_size")} @ {o.get("price")}  {o.get("asset_id","")[:20]}...')
except Exception as e:
    print(f'Error: {e}')
finally:
    pm.close()

In [ ]:
# ── Cell 10: Cancel all orders (emergency) ─────────────────
from predx import PolymarketClient
pm = PolymarketClient()
try:
    result = pm.cancel_all()
    print('All orders cancelled.', result)
except Exception as e:
    print(f'Error: {e}')
finally:
    pm.close()

In [ ]:
# ── Cell 11: Gasless merge all positions ────────────────────
import os
from dotenv import load_dotenv
load_dotenv()

from py_clob_client.client import ClobClient
from py_clob_client.clob_types import BalanceAllowanceParams
from py_builder_relayer_client.client import RelayClient
from py_builder_relayer_client.models import SafeTransaction, OperationType
from py_builder_signing_sdk.config import BuilderConfig
from py_builder_signing_sdk.sdk_types import BuilderApiKeyCreds
from web3 import Web3

pk = os.getenv('POLYMARKET_PRIVATE_KEY')
funder = os.getenv('POLYMARKET_FUNDER')

clob = ClobClient('https://clob.polymarket.com', key=pk, chain_id=137,
                   signature_type=1, funder=funder)
creds = clob.create_or_derive_api_creds()
clob.set_api_creds(creds)

builder_config = BuilderConfig(
    local_builder_creds=BuilderApiKeyCreds(
        key=os.getenv('POLY_BUILDER_API_KEY'),
        secret=os.getenv('POLY_BUILDER_SECRET'),
        passphrase=os.getenv('POLY_BUILDER_PASSPHRASE'),
    )
)
relay = RelayClient('https://relayer-v2.polymarket.com', 137, pk, builder_config)

CTF = '0x4D97DCd97eC945f40cF65F87097ACe5EA0476045'
USDC = '0x2791Bca1f2de4661ED88A30C99A7a9449Aa84174'
merge_abi = [{'name':'mergePositions','type':'function','inputs':[{'name':'collateralToken','type':'address'},{'name':'parentCollectionId','type':'bytes32'},{'name':'conditionId','type':'bytes32'},{'name':'partition','type':'uint256[]'},{'name':'amount','type':'uint256'}],'outputs':[]}]
w3 = Web3()
ctf = w3.eth.contract(address=CTF, abi=merge_abi)

print(f'{"Market":<35} {"YES":>10} {"NO":>10} {"Mergeable":>10}')
print('─' * 70)

for ms in runner.states.values():
    yb = clob.get_balance_allowance(BalanceAllowanceParams(asset_type='CONDITIONAL', token_id=ms.pair.yes_token_id))
    nb = clob.get_balance_allowance(BalanceAllowanceParams(asset_type='CONDITIONAL', token_id=ms.pair.no_token_id))
    yes_shares = float(yb['balance']) / 1e6
    no_shares = float(nb['balance']) / 1e6
    mergeable = min(yes_shares, no_shares)
    
    label = ms.pair.label[:34]
    print(f'{label:<35} {yes_shares:>10.1f} {no_shares:>10.1f} {mergeable:>10.1f}')
    
    if mergeable >= 1.0:
        amount = int(mergeable * 1e6)
        data = ctf.encode_abi('mergePositions', args=[USDC, bytes(32), ms.pair.condition_id, [1, 2], amount])
        tx = SafeTransaction(to=CTF, data=data, value='0', operation=OperationType.Call)
        try:
            resp = relay.execute([tx], 'Merge tokens to USDCe')
            print(f'  → merged {mergeable:.1f} shares  tx_id: {resp.transaction_id}')
            ms.yes.position = max(0, ms.yes.position - mergeable)
            ms.no.position = max(0, ms.no.position - mergeable)
            ms.merge_total += mergeable
        except Exception as e:
            print(f'  → merge failed: {e}')